## 🌼️ Dandelion Image Classification

Given *images of grass*, let's try to predict whether there are **dandelions** in a given image.

We will use a TensorFlow/Keras pretrained convolutional neural network to make our predictions.

Data source: https://www.kaggle.com/datasets/coloradokb/dandelionimages

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import os.path

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import tensorflow as tf

from sklearn.metrics import accuracy_score, f1_score

I0000 00:00:1792982697.022060   12151 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792982697.447961   12151 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792982700.093314   12151 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
image_dir = Path('archive/Images')

### Creating File DataFrame

In [5]:
filepaths = list(image_dir.glob(r'**/*.jpg'))
filepaths[:10]

[PosixPath('archive/Images/other/IMG_5790.jpg'),
 PosixPath('archive/Images/other/IMG_5980.jpg'),
 PosixPath('archive/Images/other/IMG_5621.jpg'),
 PosixPath('archive/Images/other/IMG_6232.jpg'),
 PosixPath('archive/Images/other/IMG_6193.jpg'),
 PosixPath('archive/Images/other/IMG_5918.jpg'),
 PosixPath('archive/Images/other/IMG_5806.jpg'),
 PosixPath('archive/Images/other/IMG_5664.jpg'),
 PosixPath('archive/Images/other/IMG_6237.jpg'),
 PosixPath('archive/Images/other/IMG_5720.jpg')]

In [12]:
labels = list(map(lambda x: os.path.split(os.path.split(x)[0])[1], filepaths))

In [19]:
filepaths = pd.Series(filepaths, name='Filepath').astype('str')
labels = pd.Series(labels, name='Label')

image_df = pd.concat([filepaths, labels], axis=1)

image_df

,Filepath,Label
0,archive/Images/other/IMG_5790.jpg,other
1,archive/Images/other/IMG_5980.jpg,other
2,archive/Images/other/IMG_5621.jpg,other
3,archive/Images/other/IMG_6232.jpg,other
4,archive/Images/other/IMG_6193.jpg,other
...,...,...
1257,archive/Images/dandelion/IMG_5529.jpg,dandelion
1258,archive/Images/dandelion/IMG_6692.jpg,dandelion
1259,archive/Images/dandelion/IMG_3931.jpg,dandelion
1260,archive/Images/dandelion/IMG_6097.jpg,dandelion


In [20]:
train_df, test_df = train_test_split(image_df, train_size=0.7, shuffle=True, random_state=1)

In [21]:
train_df

,Filepath,Label
491,archive/Images/other/IMG_5890.jpg,other
1091,archive/Images/dandelion/IMG_1216.jpg,dandelion
512,archive/Images/other/IMG_5624.jpg,other
919,archive/Images/dandelion/IMG_3861.jpg,dandelion
741,archive/Images/dandelion/IMG_1310.jpg,dandelion
...,...,...
715,archive/Images/dandelion/IMG_5930.jpg,dandelion
905,archive/Images/dandelion/IMG_1185.jpg,dandelion
1096,archive/Images/dandelion/IMG_3951.jpg,dandelion
235,archive/Images/other/IMG_6821.jpg,other


### Creating Generators

In [22]:
train_generator = ImageDataGenerator(
    preprocessing_function = tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split = 0.2
)

test_generator = ImageDataGenerator(
    preprocessing_function = tf.keras.applications.mobilenet_v2.preprocess_input 
)

In [24]:
train_images = train_generator.flow_from_dataframe(
    dataframe = train_df,
    x_col = 'Filepath',
    y_col = 'Label',
    target_size = (224, 224),
    color_mode = 'rgb',
    class_mode = 'binary',
    batch_size = 32,
    shuffle = True,
    seed = 42,
    subset = 'training'
)

val_images = train_generator.flow_from_dataframe(
    dataframe = train_df,
    x_col = 'Filepath',
    y_col = 'Label',
    target_size = (224, 224),
    color_mode = 'rgb',
    class_mode = 'binary',
    batch_size = 32,
    shuffle = True,
    seed = 42,
    subset = 'validation'
)

test_images = test_generator.flow_from_dataframe(
    dataframe = test_df,
    x_col = 'Filepath',
    y_col = 'Label',
    target_size = (224, 224),
    color_mode = 'rgb',
    class_mode = 'binary',
    batch_size = 32,
    shuffle = False
)

Found 707 validated image filenames belonging to 2 classes.
Found 176 validated image filenames belonging to 2 classes.
Found 379 validated image filenames belonging to 2 classes.


#### Downloading the Feature Extractor

In [26]:
feature_extractor = tf.keras.applications.MobileNetV2(
    input_shape = (224, 224, 3),
    weights = 'imagenet',
    include_top = False,
    pooling = 'avg'
)

feature_extractor.trainable = False

E0000 00:00:1792984164.627138   12151 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 8s 1us/step


### Training

In [28]:
inputs = feature_extractor.input
x = tf.keras.layers.Dense(128, activation='relu')(feature_extractor.output)
x = tf.keras.layers.Dense(128, activation='relu')(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,438,593 (9.30 MB)

 Trainable params: 180,609 (705.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [29]:
history = model.fit(
    train_images,
    validation_data = val_images,
    epochs = 100,
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor = 'val_loss',
            patience = 3,
            restore_best_weights = True
        )
    ]
)

Epoch 1/100


I0000 00:00:1792984542.352730   12151 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
W0000 00:00:1792984549.313529   15683 cpu_allocator_impl.cc:82] Allocation of 51380224 exceeds 10% of free system memory.
W0000 00:00:1792984549.462812   15683 cpu_allocator_impl.cc:82] Allocation of 51380224 exceeds 10% of free system memory.
W0000 00:00:1792984549.586147   15683 cpu_allocator_impl.cc:82] Allocation of 154140672 exceeds 10% of free system memory.
W0000 00:00:1792984549.764276   15683 cpu_allocator_impl.cc:82] Allocation of 156905472 exceeds 10% of free system memory.
W0000 00:00:1792984549.896897   15683 cpu_allocator_impl.cc:82] Allocation of 38535168 exceeds 10% of free system memory.


23/23 ━━━━━━━━━━━━━━━━━━━━ 98s 4s/step - accuracy: 0.6421 - loss: 0.6174 - val_accuracy: 0.8125 - val_loss: 0.4239
Epoch 2/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 85s 4s/step - accuracy: 0.8204 - loss: 0.3873 - val_accuracy: 0.8295 - val_loss: 0.3419
Epoch 3/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 85s 4s/step - accuracy: 0.8727 - loss: 0.3006 - val_accuracy: 0.8750 - val_loss: 0.2833
Epoch 4/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 141s 4s/step - accuracy: 0.9095 - loss: 0.2429 - val_accuracy: 0.8920 - val_loss: 0.2675
Epoch 5/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 88s 4s/step - accuracy: 0.9222 - loss: 0.1966 - val_accuracy: 0.8693 - val_loss: 0.2541
Epoch 6/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 84s 4s/step - accuracy: 0.9109 - loss: 0.1928 - val_accuracy: 0.8693 - val_loss: 0.2917
Epoch 7/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 110s 5s/step - accuracy: 0.9335 - loss: 0.1548 - val_accuracy: 0.8920 - val_loss: 0.2716
Epoch 8/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 90s 4s/step - accuracy: 0.9576 - loss: 0.1232 - val_accuracy: 0.8750 - val_loss: 0

### Results

In [32]:
predictions = np.squeeze(model.predict(test_images))

12/12 ━━━━━━━━━━━━━━━━━━━━ 39s 3s/step


In [37]:
predictions[:10]

array([5.2693096e-04, 9.0167803e-01, 9.2139244e-02, 9.8426628e-01,
       9.9847686e-01, 9.6988821e-01, 8.5868382e-01, 3.7712906e-02,
       8.2547700e-01, 4.2394382e-01], dtype=float32)

In [40]:
predictions = (predictions > 0.5).astype(int)
predictions[:10]

array([0, 1, 0, 1, 1, 1, 1, 0, 1, 0])

In [41]:
acc = accuracy_score(test_images.labels, predictions)

f1 = f1_score(test_images.labels, predictions)

In [42]:
print("Accuracy: {:.2f}%".format(acc*100))
print("F1-Score: {:.5f}".format(f1))

Accuracy: 86.28%
F1-Score: 0.86022
